In [1]:
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126 --quiet
!pip install torch-geometric --quiet
!pip install matplotlib seaborn scipy tqdm --quiet
import glob, sys, torch

# Find actual wheel locations
wheels = glob.glob('C:\AIML\GNN-Project\EdgeConv/**/*.whl', recursive=True)
for w in wheels:
    print(w)

print(f"\nPython: {sys.version}")
print(f"Torch:  {torch.__version__}")

import subprocess

# Detect PyTorch and CUDA versions
torch_version = torch.__version__.split('+')[0]  # "2.8.0"
cuda_version = torch.version.cuda.replace('.', '')  # "126" from "12.6"

print(f"PyTorch: {torch_version}, CUDA: {cuda_version}")

# Install PyG extensions for your exact versions
pyg_url = f"https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_version}.html"

packages = [
    'torch-scatter',
    'torch-sparse', 
    'torch-cluster',
    'torch-spline-conv'
]

for pkg in packages:
    print(f"\nInstalling {pkg}...")
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, 
         '-f', pyg_url, '--no-cache-dir', '-q'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✓ {pkg} installed successfully")
    else:
        print(f"  ✗ {pkg} failed: {result.stderr[:300]}")


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


C:\AIML\GNN-Project\EdgeConv\data\wheels\torch_cluster-1.6.3-cp312-cp312-linux_x86_64.whl
C:\AIML\GNN-Project\EdgeConv\data\wheels\torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl
C:\AIML\GNN-Project\EdgeConv\data\wheels\torch_sparse-0.6.18-cp312-cp312-linux_x86_64.whl
C:\AIML\GNN-Project\EdgeConv\data\wheels\torch_spline_conv-1.2.2-cp312-cp312-linux_x86_64.whl

Python: 3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]
Torch:  2.8.0+cu126
PyTorch: 2.8.0, CUDA: 126

Installing torch-scatter...
  ✓ torch-scatter installed successfully

Installing torch-sparse...
  ✓ torch-sparse installed successfully

Installing torch-cluster...
  ✓ torch-cluster installed successfully

Installing torch-spline-conv...
  ✓ torch-spline-conv installed successfully


In [ ]:
import os, torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch, Data
from torch_geometric.nn import global_max_pool
from torch_cluster import knn_graph
from tqdm.auto import trange, tqdm
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from scipy.spatial import cKDTree
 
# ─────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CACHE_PATH = 'C:\AIML\GNN-Project\EdgeConv\data\modelnet40_final.pt'
MOTIF_CACHE = 'C:\AIML\GNN-Project\EdgeConv\data\modelnet40_with_motifs.pt'
 
K_NEIGHBORS = 20
BATCH_SIZE = 64
EPOCHS = 200
LR = 8e-4
NUM_CLASSES = 40
TRAIN_SAMPLES_PER_CLASS = 200   # change as needed
USE_SUBSAMPLING = False
EVAL_EVERY = 5
PATIENCE = 40
WEIGHT_DECAY = 5e-5
DROPOUT = 0.3
LABEL_SMOOTHING = 0.1
SAVE_DIR = "/kaggle/working/"

os.makedirs(SAVE_DIR, exist_ok=True)

history = {
    'epoch': [],
    'train_acc': [],
    'test_oa': [],
    'macc': [],
    'loss': []
}
# ─────────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────────
print("\nLoading data...")
import torch_geometric.data.data
torch.serialization.add_safe_globals([torch_geometric.data.data.DataEdgeAttr])
 
if os.path.exists(MOTIF_CACHE):
    cache = torch.load(MOTIF_CACHE, map_location='cpu', weights_only=False)
    HAS_MOTIFS = False
elif os.path.exists(CACHE_PATH):
    cache = torch.load(CACHE_PATH, map_location='cpu', weights_only=False)
    HAS_MOTIFS = False
else:
    raise FileNotFoundError("Cache not found")
 
train_list = cache['train']
test_list = cache['test']
CLASSES = cache['classes']
print(f"✓ Loaded {len(train_list)} train + {len(test_list)} test samples")
 
# ─────────────────────────────────────────────────────
# MOTIF COMPUTATION (IF NEEDED)
# ─────────────────────────────────────────────────────
def compute_triangle_counts(edge_index: torch.Tensor, num_nodes: int) -> torch.Tensor:
    """Fast triangle counting via A² trick"""
    N = num_nodes
    src, dst = edge_index[0], edge_index[1]
    
    A = torch.zeros(N, N, dtype=torch.float32)
    A[src, dst] = 1.0
    A2 = A @ A
    
    counts = (A * A2).sum(dim=1) / 2.0
    max_c = counts.max()
    if max_c > 0:
        counts = counts / max_c
    
    return counts.unsqueeze(1)
 
 
def add_motif_features(data_list: list, k: int = 20, desc: str = '') -> list:
    """Add triangle motif scores to node features"""
    for data in tqdm(data_list, desc=f'Motif scores {desc}', leave=False):
        pos = data.pos.detach().cpu().numpy()
        N = pos.shape[0]
        
        tree = cKDTree(pos)
        _, nn_idx = tree.query(pos, k=k + 1)
        nn_idx = nn_idx[:, 1:]
        
        src = np.repeat(np.arange(N), k)
        dst = nn_idx.flatten()
        
        edge_index_geo = torch.tensor(
            np.stack([src, dst], axis=0),
            dtype=torch.long
        )
        
        tri_counts = compute_triangle_counts(edge_index_geo, N)
        data.x = torch.cat([data.x, tri_counts], dim=1)
        data.edge_index = edge_index_geo
    
    return data_list
 
 
if not HAS_MOTIFS:
    print("\nComputing motifs...")
    train_list = add_motif_features(train_list, k=K_NEIGHBORS, desc='train')
    test_list = add_motif_features(test_list, k=K_NEIGHBORS, desc='test')
    HAS_MOTIFS = True


Loading data...
✓ Loaded 9843 train + 2468 test samples

Computing motifs...


Motif scores test:  64%|██████▍   | 1589/2468 [00:15<00:07, 123.16it/s] 

In [ ]:
print("\nPrecomputing k-NN graphs...")
 
def add_cached_graph(data, k=20):
    edge_index = knn_graph(data.pos, k=k, loop=False)
    data.edge_index = edge_index
    return data
 
if not hasattr(train_list[0], 'edge_index'):
    train_list = [add_cached_graph(d, k=K_NEIGHBORS) for d in tqdm(train_list, desc='Train')]
    test_list = [add_cached_graph(d, k=K_NEIGHBORS) for d in tqdm(test_list, desc='Test')]
 
# ─────────────────────────────────────────────────────
# DATALOADERS
# ─────────────────────────────────────────────────────
def stratified_subsample(data_list, samples_per_class=100, seed=42):
    np.random.seed(seed)
    class_to_indices = {}
    for idx, data in enumerate(data_list):
        label = data.y.item() if isinstance(data.y, torch.Tensor) else data.y
        class_to_indices.setdefault(label, []).append(idx)

    keep_indices = []
    for label, indices in class_to_indices.items():
        if len(indices) > samples_per_class:
            keep = np.random.choice(indices, samples_per_class, replace=False)
        else:
            keep = indices
        keep_indices.extend(keep)

    return [data_list[i] for i in keep_indices]

def augment(batch):
    device = batch.pos.device
    B = batch.num_graphs
    
    # Build all B rotation matrices at once — no Python loop, no numpy
    theta = torch.rand(B, device=device) * 2 * np.pi
    cos_t = torch.cos(theta)
    sin_t = torch.sin(theta)
    zeros = torch.zeros(B, device=device)
    ones  = torch.ones(B, device=device)
    
    # [B, 3, 3] batch of Y-axis rotation matrices
    R = torch.stack([
        torch.stack([ cos_t, zeros, sin_t], dim=1),
        torch.stack([ zeros,  ones, zeros], dim=1),
        torch.stack([-sin_t, zeros, cos_t], dim=1),
    ], dim=1)  # [B, 3, 3]
    
    # Gather per-node rotation matrix using batch index
    R_per_node = R[batch.batch]   # [N, 3, 3]
    
    # Rotate all nodes in one bmm
    pts = torch.bmm(batch.pos.unsqueeze(1), R_per_node.transpose(1,2)).squeeze(1)
    
    # Scale and jitter — also vectorised
    scale = 0.8 + torch.rand(B, device=device) * 0.45
    pts   = pts * scale[batch.batch].unsqueeze(1)
    pts   = pts + (torch.randn_like(pts) * 0.02).clamp(-0.05, 0.05)
    
    batch.pos = pts
    batch.x[:, :3]  = pts
    batch.x[:, 3:6] = torch.bmm(
        batch.x[:, 3:6].unsqueeze(1), R_per_node.transpose(1,2)
    ).squeeze(1)
    return batch

def downsample_points(data, num_points=512):
    if data.pos.shape[0] > num_points:
        idx = torch.randperm(data.pos.shape[0])[:num_points]
        data.pos = data.pos[idx]
        data.x   = data.x[idx]
    return data

if USE_SUBSAMPLING:
    print(f"Subsampling training set to {TRAIN_SAMPLES_PER_CLASS} samples per class...")
    train_list_subsampled = stratified_subsample(train_list, samples_per_class=TRAIN_SAMPLES_PER_CLASS)
    print(f"Original train size: {len(train_list)}, after subsampling: {len(train_list_subsampled)}")
    train_loader = DataLoader(train_list_subsampled, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
else:
    train_loader = DataLoader(train_list, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
    
test_loader = DataLoader(test_list, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
print(train_list[0].x[:5])

In [ ]:
from torch_geometric.nn import EdgeConv
from torch_geometric.nn import global_max_pool, global_mean_pool
from torch.amp import autocast, GradScaler

def make_edgeconv_block(in_channels, out_channels, dropout=0.5):
    """Create EdgeConv block with dropout"""
    return EdgeConv(
        nn.Sequential(
            nn.Linear(in_channels * 2, out_channels, bias=False),
            nn.BatchNorm1d(out_channels),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout * 0.5),
            nn.Linear(out_channels, out_channels, bias=False),
            nn.BatchNorm1d(out_channels),
            nn.LeakyReLU(0.2)
        ),
        aggr='max'
    )
 
def per_graph_topk(values, batch, keep_ratio):
    """
    Keep top-k nodes per graph where k = max(1, int(num_nodes * keep_ratio)).
    
    Args:
        values: [N] tensor of scores (higher = more important)
        batch:  [N] graph assignment for each node
        keep_ratio: float in (0,1], fraction of nodes to keep per graph
    
    Returns:
        keep_idx: global indices of kept nodes
    """
    device = values.device
    unique_graphs = batch.unique()
    keep_idx = []
    for g in unique_graphs:
        mask = (batch == g)
        vals = values[mask]
        n = vals.size(0)
        k = max(1, int(n * keep_ratio))
        k = min(k, n)  # safety
        if k == n:
            idx = torch.arange(n, device=device)
        else:
            _, idx = torch.topk(vals, k)
        global_idx = torch.where(mask)[0][idx]
        keep_idx.append(global_idx)
    return torch.cat(keep_idx)
    
# ══════════════════════════════════════════════════════════════════════════════
# METHOD 1: MC DROPOUT UNCERTAINTY-BASED SUBGRAPHING
# ══════════════════════════════════════════════════════════════════════════════
 
class BaselineDGCNN(nn.Module):
    """
    Standard DGCNN as per Wang et al. 2019.
    - 4 EdgeConv layers with output channels 64, 64, 128, 256
    - Dynamic graph construction (k‑NN in feature space)
    - Global max + avg pooling → classifier
    """
    def __init__(self, in_channels=6, num_classes=40, k=20, dropout=0.5, small=False):
        super().__init__()
        self.k = k
        if small:
            # Smaller model: reduce channels to speed up training
            ch1, ch2, ch3, ch4 = 32, 32, 64, 128
            embed_dims = 512
        else:
            ch1, ch2, ch3, ch4 = 64, 64, 128, 256
            embed_dims = 1024

        # EdgeConv layers (single linear + BN + LeakyReLU)
        self.conv1 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * in_channels, ch1, bias=False),
                nn.BatchNorm1d(ch1),
                nn.LeakyReLU(0.2)
            ),
            aggr='max'
        )
        self.conv2 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * ch1, ch2, bias=False),
                nn.BatchNorm1d(ch2),
                nn.LeakyReLU(0.2)
            ),
            aggr='max'
        )
        self.conv3 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * ch2, ch3, bias=False),
                nn.BatchNorm1d(ch3),
                nn.LeakyReLU(0.2)
            ),
            aggr='max'
        )
        self.conv4 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * ch3, ch4, bias=False),
                nn.BatchNorm1d(ch4),
                nn.LeakyReLU(0.2)
            ),
            aggr='max'
        )

        # Global aggregation
        total_channels = ch1 + ch2 + ch3 + ch4
        self.global_mlp = nn.Sequential(
            nn.Linear(total_channels*2, embed_dims, bias=False),
            nn.BatchNorm1d(embed_dims),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Classifier
        if small:
            self.classifier = nn.Sequential(
                nn.Linear(embed_dims, 256, bias=False),
                nn.BatchNorm1d(256),
                nn.LeakyReLU(0.2),
                nn.Dropout(dropout),
                nn.Linear(256, 128, bias=False),
                nn.BatchNorm1d(128),
                nn.LeakyReLU(0.2),
                nn.Dropout(dropout),
                nn.Linear(128, num_classes)
            )
        else:
            self.classifier = nn.Sequential(
                nn.Linear(embed_dims, 512, bias=False),
                nn.BatchNorm1d(512),
                nn.LeakyReLU(0.2),
                nn.Dropout(dropout),
                nn.Linear(512, 256, bias=False),
                nn.BatchNorm1d(256),
                nn.LeakyReLU(0.2),
                nn.Dropout(dropout),
                nn.Linear(256, num_classes)
            )
    def forward(self, data):
        x = data.x[:, :6]          # pos + norm only
        batch = data.batch
        pos = data.pos

        # Dynamic graph at each layer
        edge_index = knn_graph(pos, k=self.k, batch=batch, loop=False)
        x1 = self.conv1(x, edge_index)

        edge_index = knn_graph(x1, k=self.k, batch=batch, loop=False)
        x2 = self.conv2(x1, edge_index)

        edge_index = knn_graph(x2, k=self.k, batch=batch, loop=False)
        x3 = self.conv3(x2, edge_index)

        edge_index = knn_graph(x3, k=self.k, batch=batch, loop=False)
        x4 = self.conv4(x3, edge_index)

        # Concatenate multi-scale features
        x = torch.cat([x1, x2, x3, x4], dim=1)

        # Global max + average pooling
        x_max = global_max_pool(x, batch)
        x_avg = global_mean_pool(x, batch)
        x = torch.cat([x_max, x_avg], dim=1)

        # Classifier
        x = self.global_mlp(x)
        x = self.classifier(x)
        return x

In [ ]:
import json
import pandas as pd
def train_epoch(model, loader, optimizer, device, scaler, grad_clip=1.0, label_smoothing=0.1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch in tqdm(loader, desc='  train', leave=False):
        batch = batch.to(device, non_blocking=True)
        batch = augment(batch)                         # FIX 1: uncommented

        optimizer.zero_grad(set_to_none=True)

        with autocast('cuda'):
            out  = model(batch)
            loss = F.cross_entropy(out, batch.y.squeeze(), label_smoothing=label_smoothing)

        scaler.scale(loss).backward()

        if grad_clip is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * batch.num_graphs
        correct    += out.argmax(1).eq(batch.y.squeeze()).sum().item()
        total      += batch.num_graphs

    return total_loss / total, correct / total


@torch.no_grad()
def test_epoch(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc='  test ', leave=False):
        batch = batch.to(device, non_blocking=True)
        with autocast('cuda'):
            preds = model(batch).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch.y.squeeze().cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    return {
        'OA':       accuracy_score(all_labels, all_preds) * 100,
        'mAcc':     balanced_accuracy_score(all_labels, all_preds) * 100,
        'macro_f1': f1_score(all_labels, all_preds, average='macro', zero_division=0) * 100,
    }


def train_model(model, model_name, train_loader, test_loader, device,
                epochs=EPOCHS, patience=PATIENCE):   # FIX 3: pull from config
    print(f"\n{'='*80}")
    print(f"TRAINING: {model_name}  |  AMP: ON  |  device: {device}")
    print(f"epochs={epochs}  patience={patience}  eval_every={EVAL_EVERY}  lr={LR}")
    print(f"dropout={DROPOUT}  label_smoothing={LABEL_SMOOTHING}  weight_decay={WEIGHT_DECAY}")
    print(f"{'='*80}")

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-5)
    scaler    = torch.amp.GradScaler('cuda')

    history = {
        'epoch': [], 'train_loss': [], 'train_acc': [],
        'test_oa': [], 'test_macc': [], 'test_f1': []
    }

    best_oa          = 0.0
    patience_counter = 0
    metrics          = {'OA': 0.0, 'mAcc': 0.0, 'macro_f1': 0.0}  # FIX 2: initialise

    for epoch in trange(1, epochs + 1, desc=model_name):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, device, scaler)
        scheduler.step()                               # FIX 4: always step, every epoch

        # Evaluate on schedule
        if epoch % EVAL_EVERY == 0 or epoch == epochs:
            metrics = test_epoch(model, test_loader, device)
            if metrics['OA'] > best_oa:
                best_oa          = metrics['OA']
                patience_counter = 0
                torch.save(model.state_dict(),
                           os.path.join(SAVE_DIR, f'best_{model_name}.pth'))
            else:
                patience_counter += 1

        # Log every epoch (metrics holds last eval values on non-eval epochs)
        history['epoch'].append(epoch)
        history['train_loss'].append(round(tr_loss, 4))
        history['train_acc'].append(round(tr_acc * 100, 2))
        history['test_oa'].append(round(metrics['OA'], 2))
        history['test_macc'].append(round(metrics['mAcc'], 2))
        history['test_f1'].append(round(metrics['macro_f1'], 2))

        if epoch % 10 == 0:
            print(f"\n  Epoch {epoch:3d} | loss {tr_loss:.4f} | "
                  f"trAcc {tr_acc*100:.1f}% | OA {metrics['OA']:.1f}% | "
                  f"mAcc {metrics['mAcc']:.1f}% | "
                  f"gap {(tr_acc*100 - metrics['OA']):+.1f}%")
            if device.type == 'cuda':
                print(f"            VRAM: "
                      f"{torch.cuda.memory_allocated()/1e9:.2f}/"
                      f"{torch.cuda.memory_reserved()/1e9:.2f} GB")
            if hasattr(model, 'alpha'):
                a = torch.sigmoid(model.alpha).item()
                print(f"            α={a:.3f} (unc {a:.0%} / motif {1-a:.0%})")

        if patience_counter >= patience:
            print(f"\n  Early stopping at epoch {epoch}")
            break

    history_path = os.path.join(SAVE_DIR, f'history_{model_name}.json')
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)
    pd.DataFrame(history).to_csv(
        os.path.join(SAVE_DIR, f'history_{model_name}.csv'), index=False)

    print(f"\n{'='*80}")
    print(f"  {model_name}  →  Best OA: {best_oa:.2f}%")
    print(f"  History: {history_path}")
    print(f"{'='*80}")
    return best_oa, history

In [ ]:
import time

model = BaselineDGCNN(in_channels=6, num_classes=NUM_CLASSES, k=K_NEIGHBORS, dropout=0.5, small=True).to(DEVICE)
scaler = torch.amp.GradScaler()
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

best_oa = 0.0
patience = 20
patience_counter = 0

def debug_train_epoch(model, loader, optimizer, device, scaler):
    """Debug version with timing breakdown."""
    model.train()
    
    # Timing accumulators
    data_time = 0
    forward_time = 0
    backward_time = 0
    graph_time = 0
    other_time = 0
    
    for i, batch in enumerate(loader):
        if i >= 5:  # Only profile first 5 batches
            break
            
        # Time data transfer
        t0 = time.time()
        batch = batch.to(device, non_blocking=True)
        data_time += time.time() - t0
        
        optimizer.zero_grad(set_to_none=True)
        
        # Time forward pass (including graph construction)
        t1 = time.time()
        with autocast('cuda'):
            out = model(batch)
            loss = F.cross_entropy(out, batch.y.squeeze())
        forward_time += time.time() - t1
        
        # Time backward pass
        t2 = time.time()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        backward_time += time.time() - t2
        
        # If your model has custom graph ops, add this
        if hasattr(model, 'get_graph_time'):
            graph_time += model.get_graph_time()
        
        print(f"Batch {i}: data={data_time:.3f}s, forward={forward_time:.3f}s, backward={backward_time:.3f}s")
    
    print("\n" + "="*50)
    print("TIMING BREAKDOWN (first 5 batches):")
    print(f"  Data transfer: {data_time:.2f}s ({data_time/(data_time+forward_time+backward_time)*100:.1f}%)")
    print(f"  Forward pass:  {forward_time:.2f}s ({forward_time/(data_time+forward_time+backward_time)*100:.1f}%)")
    print(f"  Backward pass: {backward_time:.2f}s ({backward_time/(data_time+forward_time+backward_time)*100:.1f}%)")
    print("="*50)

debug_train_epoch(model=model, loader=train_loader, optimizer=optimizer, device=DEVICE, scaler=scaler)

In [ ]:
# Quick sanity check — run before full training
print(f"Train samples : {len(train_list)}")        # should be ~9840
print(f"Test samples  : {len(test_list)}")         # should be 2468
print(f"Points/sample : {train_list[0].pos.shape}")# should be [1024, 3]
print(f"Features/node : {train_list[0].x.shape}")  # should be [1024, 7]
print(f"LR            : {LR}")                     # must be 0.001
print(f"Batch size    : {BATCH_SIZE}")              # 32
print(f"Subsampling   : {USE_SUBSAMPLING}")        # False

# Time one epoch to estimate total runtime
import time
model_test = BaselineDGCNN(
    in_channels=6, num_classes=40, k=K_NEIGHBORS, dropout=0.5, small=True
).to(DEVICE)
opt_test = torch.optim.Adam(model_test.parameters(), lr=LR)
scaler_test = GradScaler('cuda')

t0 = time.time()
train_epoch(model_test, train_loader, opt_test, DEVICE, scaler_test)
ep_time = time.time() - t0
print(f"\nTime per epoch : {ep_time:.1f}s")
print(f"Est. 200 epochs: {ep_time*200/3600:.1f} hours")
print(f"Est. 150 epochs: {ep_time*150/3600:.1f} hours")
del model_test, opt_test, scaler_test
torch.cuda.empty_cache()

In [ ]:
best_oa = train_model(model=model, model_name='DGCNN', train_loader=train_loader, test_loader=test_loader, device=DEVICE, epochs=200)


In [ ]:
# ── Checkpoint: save final weights & reload best ──────────────────────────
import os

best_oa, history = best_oa  # unpack if train_model returned a tuple

# Save final-epoch weights (separate from the mid-training best checkpoint)
final_path = os.path.join(SAVE_DIR, 'final_DGCNN.pth')
torch.save({
    'epoch':       len(history['epoch']),
    'model_state': model.state_dict(),
    'best_oa':     best_oa,
    'history':     history,
    'config': {
        'k':          K_NEIGHBORS,
        'in_channels': 6,
        'num_classes': NUM_CLASSES,
        'small':      True,
        'dropout':    DROPOUT,
    },
}, final_path)
print(f'Final weights saved  : {final_path}')

# Reload the best checkpoint so downstream cells (profiling, plotting)
# operate on the best model, not the last epoch
best_path = os.path.join(SAVE_DIR, 'best_DGCNN.pth')
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()
print(f'Best weights loaded  : {best_path}')
print(f'Best OA achieved     : {best_oa:.2f}%')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EFFICIENCY PROFILING — FLOPs, Latency, Throughput, Memory
# ══════════════════════════════════════════════════════════════════════════════
import json, time, gc, os
import torch
import torch.profiler
import numpy as np

# ─── helpers ───────────────────────────────────────────────────────────────

def count_parameters(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def measure_flops(model, sample, device, n_warmup=3, n_measure=10):
    """
    Measure FLOPs via torch.profiler ATen-op tracing.
    knn_graph (C++ extension) is NOT counted — standard GNN-paper practice;
    note this exclusion when reporting.
    Returns dict with flops_per_forward and a human-readable string.
    """
    model.eval()
    data = sample.clone().to(device)

    def run_once():
        with torch.no_grad():
            model(data)

    # Warm-up (fills CUDA caches, JIT compiles kernels)
    for _ in range(n_warmup):
        run_once()
    if device.type == 'cuda':
        torch.cuda.synchronize()

    activities = [torch.profiler.ProfilerActivity.CPU]
    if device.type == 'cuda':
        activities.append(torch.profiler.ProfilerActivity.CUDA)

    with torch.profiler.profile(
        activities=activities,
        with_flops=True,
        record_shapes=True,
    ) as prof:
        for _ in range(n_measure):
            run_once()
        if device.type == 'cuda':
            torch.cuda.synchronize()

    total_flops = sum(
        e.flops for e in prof.key_averages() if e.flops is not None
    )
    per_forward = total_flops / n_measure
    gflops = per_forward / 1e9
    return {
        'flops_per_forward': per_forward,
        'gflops_per_forward': gflops,
        'flops_str': f'{gflops:.4f} GFLOPs  (knn_graph excluded)',
        'profiler_note': 'ATen-op tracing via torch.profiler; knn_graph C++ calls not counted',
    }


def measure_latency(model, sample, device, n_warmup=5, n_measure=50):
    """
    Wall-clock latency per single-sample forward pass.
    Uses CUDA events for sub-millisecond GPU precision.
    """
    model.eval()
    data = sample.clone().to(device)

    if device.type == 'cuda':
        for _ in range(n_warmup):
            with torch.no_grad():
                model(data)
        torch.cuda.synchronize()
        times = []
        for _ in range(n_measure):
            start_evt = torch.cuda.Event(enable_timing=True)
            end_evt   = torch.cuda.Event(enable_timing=True)
            start_evt.record()
            with torch.no_grad():
                model(data)
            end_evt.record()
            torch.cuda.synchronize()
            times.append(start_evt.elapsed_time(end_evt))   # ms
    else:
        for _ in range(n_warmup):
            with torch.no_grad():
                model(data)
        times = []
        for _ in range(n_measure):
            t0 = time.perf_counter()
            with torch.no_grad():
                model(data)
            times.append((time.perf_counter() - t0) * 1000)  # ms

    times = np.array(times)
    return {
        'latency_mean_ms': float(times.mean()),
        'latency_std_ms':  float(times.std()),
        'latency_p50_ms':  float(np.percentile(times, 50)),
        'latency_p95_ms':  float(np.percentile(times, 95)),
        'latency_p99_ms':  float(np.percentile(times, 99)),
    }


def measure_throughput(model, loader, device, n_batches=20):
    """
    Samples-per-second on real DataLoader batches (test split).
    """
    model.eval()
    total_samples = 0
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t_start = time.perf_counter()
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches:
                break
            batch = batch.to(device, non_blocking=True)
            model(batch)
            total_samples += batch.num_graphs
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t_start
    return {
        'throughput_samples_per_sec':   total_samples / elapsed,
        'throughput_batches_measured':  n_batches,
        'throughput_total_samples':     total_samples,
        'throughput_elapsed_sec':       elapsed,
    }


def measure_memory(model, sample, device):
    """
    Peak GPU memory (MiB) for a single forward pass.
    Returns 0 on CPU.
    """
    if device.type != 'cuda':
        return {'peak_memory_mib': 0.0, 'note': 'CPU — no GPU memory tracked'}
    model.eval()
    data = sample.clone().to(device)
    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize()
    with torch.no_grad():
        model(data)
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    return {'peak_memory_mib': float(peak)}


def full_profile(model_name, model, sample, loader, device):
    """Run all profilers and return a single flat results dict."""
    print(f"\n{'='*60}")
    print(f"  Profiling: {model_name}")
    print(f"{'='*60}")
    model.eval()

    total_p, train_p = count_parameters(model)
    params = {'total_params': total_p, 'trainable_params': train_p}
    print(f"  Parameters  : {total_p:,} total  |  {train_p:,} trainable")

    print('  Measuring FLOPs ...', end=' ', flush=True)
    flops = measure_flops(model, sample, device)
    print(flops['flops_str'])

    print('  Measuring latency ...', end=' ', flush=True)
    lat = measure_latency(model, sample, device)
    print(f"{lat['latency_mean_ms']:.2f} \u00b1 {lat['latency_std_ms']:.2f} ms  "
          f"(p95={lat['latency_p95_ms']:.2f} ms)")

    print('  Measuring throughput ...', end=' ', flush=True)
    tput = measure_throughput(model, loader, device)
    print(f"{tput['throughput_samples_per_sec']:.1f} samples/s")

    mem = measure_memory(model, sample, device)
    if device.type == 'cuda':
        print(f"  Peak GPU mem: {mem['peak_memory_mib']:.1f} MiB")

    return {'model': model_name, **params, **flops, **lat, **tput, **mem}


# ─── run ───────────────────────────────────────────────────────────────────

PROFILE_SAVE_PATH = 'efficiency_metrics.json'

# Use the first test sample (1024 pts) for single-sample benchmarks
sample = test_list[0]

all_metrics = {}
all_metrics['Baseline'] = full_profile(
    'Baseline', model, sample, test_loader, DEVICE
)

# Add subgraph models here once they exist, e.g.:
# all_metrics['MC Dropout'] = full_profile('MC Dropout', model_mc,  sample, test_loader, DEVICE)
# all_metrics['Motif']      = full_profile('Motif',      model_motif, sample, test_loader, DEVICE)
# all_metrics['Hybrid']     = full_profile('Hybrid',     model_hybrid, sample, test_loader, DEVICE)

# ─── comparison table ──────────────────────────────────────────────────────
print()
print('=' * 86)
print('EFFICIENCY SUMMARY')
print('=' * 86)
hdr = (f"{'Model':<18} {'Params':>10}  {'GFLOPs':>8}  "
       f"{'Lat ms':>8}  {'p95 ms':>7}  {'Tput samp/s':>12}  {'Mem MiB':>8}")
print(hdr)
print('-' * 86)
for name, m in all_metrics.items():
    gf  = m.get('gflops_per_forward', 0)
    mem = m.get('peak_memory_mib', 0)
    print(
        f"{name:<18} {m['total_params']:>10,}  {gf:>8.4f}  "
        f"{m['latency_mean_ms']:>8.2f}  {m['latency_p95_ms']:>7.2f}  "
        f"{m['throughput_samples_per_sec']:>12.1f}  {mem:>8.1f}"
    )
print('=' * 86)
print('Notes:')
print('  GFLOPs  — ATen-op tracing (torch.profiler); knn_graph C++ ext excluded.')
print('  Latency — single-sample forward pass; CUDA event timing (GPU), perf_counter (CPU).')
print('  Throughput — real test-set DataLoader batches.')
print('  Memory  — peak GPU alloc during single forward pass (torch.cuda.max_memory_allocated).')

# ─── save ──────────────────────────────────────────────────────────────────
def _to_python(v):
    if hasattr(v, 'item'):   # numpy scalar
        return v.item()
    return v

save_dict = {
    k: {kk: _to_python(vv) for kk, vv in v.items()}
    for k, v in all_metrics.items()
}
with open(PROFILE_SAVE_PATH, 'w') as f:
    json.dump(save_dict, f, indent=2)

print(f"\nAll metrics saved to: {os.path.abspath(PROFILE_SAVE_PATH)}")


In [ ]:
import matplotlib.pyplot as plt                                                                   
print("\n" + "="*80)
print(f"TRAINING COMPLETE")
print("EPOCHS_DATA =", history['epoch'])
print("TRAIN_ACC   =", history['train_acc'])
print("TEST_OA     =", history['test_oa'])
print("MACC        =", history['macc'])
print("LOSS        =", history['loss'])
print("="*80)

def plot_history(history, save_dir):
    epochs = history['epoch']

    # Accuracy plot
    plt.figure()
    plt.plot(epochs, history['train_acc'], label='Train Acc')
    plt.plot(epochs, history['test_oa'], label='Test OA')
    plt.plot(epochs, history['macc'], label='mAcc')
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.legend()
    plt.grid()
    plt.savefig(os.path.join(save_dir, "accuracy.png"))
    plt.close()

    # Loss plot
    plt.figure()
    plt.plot(epochs, history['loss'], label='Loss')
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid()
    plt.savefig(os.path.join(save_dir, "loss.png"))
    plt.close()

plot_history(history, SAVE_DIR)